# Mujoco Jacobian
- Get Jacobian of the specific body
- Get pseudo-inverse of Jacobian
    - Damped Least Squares
    - Singular Value Decomposition

#### 1. Create Environment

In [ ]:
import os
import sys
import numpy as np
import time

import mujoco

sys.path.append(os.path.abspath('../'))
from pp_base_mujoco.VIEWER import *
from pp_base_mujoco.UTILS import *

In [ ]:
xml_path = '../asset/panda_scene.xml'
xml_abs_path = os.path.abspath(xml_path)

model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

body_names = get_body_names(model, data)
print(body_names)

#### 2. Get Jacobian

In [ ]:
def get_jacobian(
        model,
        data,
        name,
        type="body",
):
    """
    Get Jacobian of given body/geom/site
    Args
        - model: mujoco model
        - data: mujoco data
        - name: name of the body/geom/site
        - type: "body", "geom", or "site"
    """
    Jacobian_p = np.zeros((3, model.nu))
    Jacobian_r = np.zeros((3, model.nu))
    if type == "body":
        mujoco.mj_jacBody(model, data, Jacobian_p, Jacobian_r, data.body(name).id)
    elif type == "geom":
        mujoco.mj_jacGeom(model, data, Jacobian_p, Jacobian_r, data.geom(name).id)
    elif type == "site":
        mujoco.mj_jacSite(model, data, Jacobian_p, Jacobian_r, data.site(name).id)
    
    return Jacobian_p, Jacobian_r

In [ ]:
""" GET MUJOCO JACOBIAN """

ee_body_name = "right_hand"

mujoco.mj_resetData(model, data)
qpos_init = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0.5]) # set initial qpos
data.qpos[:] = qpos_init
mujoco.mj_forward(model, data)

Jacobian_p, Jacobian_r = get_jacobian(model, data, ee_body_name)
jacobian = np.vstack((Jacobian_p, Jacobian_r))
print(jacobian)

### 3. Jacobian Pseudo-inverse
#### Singular Value Decomposition
- Decompose jacobian into U, V and sigma
    - V: direction of motion in joint space
    - U: corresponding motion direction in end-effector space
- Sigma: Contains singular values (degree of amplification)
    - large singular value -> small joint move makes large 
    - Thresholding sigma to avoid singular value
- $J = U \Sigma V^T$
- $J^+ = V \Sigma^+ U^T$
</br>

In [23]:
U, Sigma, V_T = np.linalg.svd(jacobian, compute_uv=True)
row, col = jacobian.shape
print("U shape:", U.shape)
print("Sigma shape:", Sigma.shape)
print("V shape:", V_T.shape)

## IMPORTANT: suppress singularities with modified sigma 
sigma_threshold = 1e-3

Sigma_clipped_rev = np.zeros_like(Sigma)
for i, value in enumerate(Sigma):
    if Sigma[i] < sigma_threshold:
        Sigma_clipped_rev[i] = 0
    else:
        Sigma_clipped_rev[i] = 1/Sigma[i]

# inverse matrix for position jacobian
S_rev_matrix = np.zeros((col, row))
for i, value in enumerate(Sigma_clipped_rev):
    S_rev_matrix[i,i] = value
inversed = V_T.T @ S_rev_matrix @ U.T

print(f"Pseudo-inverse of Jacobian:\n{inversed}")

U shape: (6, 6)
Sigma shape: (6,)
V shape: (7, 7)
Pseudo-inverse of Jacobian:
[[-1.59967325e-15  9.17456524e-01 -4.30206019e-16  2.61139279e-01
  -1.40114869e-16  1.85477563e-01]
 [ 3.34021408e+00 -1.58585999e-16 -7.38442799e-01  3.76474656e-17
   2.90749833e-01 -1.36263909e-16]
 [-1.01370782e-15  1.46675910e+00 -6.42125985e-16 -3.86633184e-01
  -4.14447377e-16 -1.66806734e-01]
 [ 2.65129010e+00  3.98811217e-16  2.02140821e+00 -6.18675789e-17
   4.60246318e-01  1.14452829e-16]
 [-5.10153508e-16  7.73346268e-01 -4.66490184e-16  8.95898474e-01
  -2.55687667e-16 -8.79485701e-02]
 [ 6.88923980e-01 -1.86982891e-16 -2.75985101e+00 -1.64048820e-17
  -1.16949648e+00  9.08205600e-17]
 [ 8.49076299e-16  1.88283313e+00 -2.60972433e-15 -4.50988577e-01
  -1.21650916e-15 -9.24309599e-01]]


### Damped Least Squares
- pseudo-inverse of Jacobian
- Damping factor: reduce singularity issue
    - lambda damping
    - e with error vector
- $\Delta\theta = J^T(JJ^T + \lambda^2I)^{-1} \vec{e}$

In [24]:
damping = 1.0 # DLS damping factor for numerical stability
inversed = jacobian.T @ np.linalg.inv(jacobian @ jacobian.T + damping**2 * np.eye(row))
print(f"Pseudo-inverse of Jacobian with DLS:\n{inversed}")

Pseudo-inverse of Jacobian with DLS:
[[-5.19949097e-17  1.75319543e-01  6.64062997e-18  1.06292569e-01
   9.17894591e-17  2.43016130e-01]
 [ 1.60834634e-01  4.42066729e-17 -1.34706627e-01  5.52700457e-17
   2.24866814e-01  1.34543191e-16]
 [-3.79537756e-17  2.06029005e-01  1.34493581e-18 -1.59949689e-01
   6.29649315e-17  1.52390750e-01]
 [ 1.38567320e-01  1.20115205e-16  2.00634908e-01  2.17492974e-17
  -1.96288316e-01  3.50506677e-17]
 [ 3.49162005e-19  1.12138557e-01 -5.12923992e-18  4.34975193e-01
  -6.47706613e-18 -3.79248870e-02]
 [ 6.12146907e-02 -8.07560836e-17 -1.30060587e-01  6.95973241e-17
  -2.78316126e-01  1.93184183e-16]
 [ 4.41753782e-17  1.54730449e-01 -1.92686631e-17 -1.07545020e-01
  -9.36916596e-17 -3.03733042e-01]]


#### 4. Function: Pseudo Inverse

In [ ]:
def get_pseudo_inverse(
        jacobian, # stacked jacobian
        method='svd',
        sigma_threshold=1e-3, # for SVD
        damping=1.0 # for DLS
        ):

    row, col = jacobian.shape
    print(f"Jacobian shape: {jacobian.shape}")

    if method=='svd':
        U, Sigma, V_T = np.linalg.svd(jacobian, compute_uv=True)
        print("U shape:", U.shape)
        print("Sigma shape:", Sigma.shape)
        print("V shape:", V_T.shape)

        # suppress singularities with modified sigma
        Sigma_clipped_rev = np.zeros_like(Sigma)
        for i, value in enumerate(Sigma):
            if Sigma[i] < sigma_threshold:
                Sigma_clipped_rev[i] = 0
            else:
                Sigma_clipped_rev[i] = 1/Sigma[i]

        # inverse matrix for position jacobian
        S_rev_matrix = np.zeros((col, row))
        for i, value in enumerate(Sigma_clipped_rev):
            S_rev_matrix[i,i] = value
        inversed = V_T.T @ S_rev_matrix @ U.T

    if method=='DLS':
        # apply damped least squares
        inversed = jacobian.T @ np.linalg.inv(jacobian @ jacobian.T + damping**2 * np.eye(row))
    
    return inversed

In [26]:
jacobian_inversed = get_pseudo_inverse(jacobian)
print(f"Pseudo-inverse of Jacobian:\n{jacobian_inversed}")

Jacobian shape: (6, 7)
U shape: (6, 6)
Sigma shape: (6,)
V shape: (7, 7)
Pseudo-inverse of Jacobian:
[[-1.59967325e-15  9.17456524e-01 -4.30206019e-16  2.61139279e-01
  -1.40114869e-16  1.85477563e-01]
 [ 3.34021408e+00 -1.58585999e-16 -7.38442799e-01  3.76474656e-17
   2.90749833e-01 -1.36263909e-16]
 [-1.01370782e-15  1.46675910e+00 -6.42125985e-16 -3.86633184e-01
  -4.14447377e-16 -1.66806734e-01]
 [ 2.65129010e+00  3.98811217e-16  2.02140821e+00 -6.18675789e-17
   4.60246318e-01  1.14452829e-16]
 [-5.10153508e-16  7.73346268e-01 -4.66490184e-16  8.95898474e-01
  -2.55687667e-16 -8.79485701e-02]
 [ 6.88923980e-01 -1.86982891e-16 -2.75985101e+00 -1.64048820e-17
  -1.16949648e+00  9.08205600e-17]
 [ 8.49076299e-16  1.88283313e+00 -2.60972433e-15 -4.50988577e-01
  -1.21650916e-15 -9.24309599e-01]]
